# Mineração de Textos — Classificação de Reclamações (demo Aula 8)

Pipeline: **dados → pré-processamento → embeddings → baseline clássico →
LLM com Pydantic (classificação + enriquecimento) → RAG (bônus)**.

Este notebook apenas **orquestra** o código de `src/` — a lógica vive nos módulos
(notebooks soltos não são o entregável). Rode da raiz do projeto.

In [ ]:
import sys, os, warnings
warnings.filterwarnings("ignore")
# garante que o pacote src/ seja importável a partir da raiz do projeto
if "src" not in os.listdir("."):
    os.chdir("..")
from src import data, preprocess, baseline, embeddings, evaluate, custos
print("módulos carregados")

## 1. Dados e EDA

In [ ]:
df = data.carregar()
print("Reclamações válidas:", len(df))
df["categoria"].value_counts()

In [ ]:
import matplotlib.pyplot as plt
ax = df["categoria"].value_counts().plot.barh(figsize=(7,4))
ax.invert_yaxis(); ax.set_title("Reclamações por categoria"); plt.tight_layout(); plt.show()

In [ ]:
# % não resolvida por setor (insight 1)
nao = ["Não resolvida", "Sem resposta da empresa"]
(df.assign(nao_res=df["status"].isin(nao))
   .groupby("categoria")["nao_res"].mean().mul(100).sort_values(ascending=False).round(1))

## 2. Pré-processamento (Aula 2)

In [ ]:
for t in df["texto"].head(3):
    print("ORIG :", t[:80])
    print("LIMPO:", preprocess.limpar(t)); print("-"*50)

## 3. Baseline clássico — ANTES do LLM (obrigatório)

In [ ]:
tr, te = data.split(df)
f1_tfidf = baseline.baseline_tfidf(tr, te)
f1_emb   = baseline.baseline_embeddings(tr, te)
print(f"\nTF-IDF={f1_tfidf:.4f} | Embeddings={f1_emb:.4f}")

## 4. LLM com Pydantic — o núcleo

Schema = contrato de saída. Requer `GEMINI_API_KEY` no `.env`. Use um limite
pequeno na demo (cache evita re-chamar a API).

In [ ]:
from src.schemas import AnaliseReclamacao
print(AnaliseReclamacao.model_json_schema().get("properties").keys())

In [ ]:
# Classificação (L1) + enriquecimento (L2). Ajuste o limite conforme a cota.
try:
    from src import classify_llm
    LIM = 30
    classify_llm.classificar(te, "zeroshot", limite=LIM)
    classify_llm.classificar(te, "fewshot", limite=LIM)
    classify_llm.enriquecer(tr, te, "zeroshot", limite=LIM)
except Exception as e:
    print("LLM não executado (configure GEMINI_API_KEY no .env):", type(e).__name__, e)

## 5. Tabela comparativa (baseline vs LLM)

In [ ]:
evaluate.gerar_relatorio()

## 6. RAG sobre o corpus (bônus)

In [ ]:
from src import rag
index = rag.construir_indice()
tabela, media = rag.avaliar_relevancia(index)
print("relevância@5 média:", media)

In [ ]:
# resposta ancorada (requer chave)
try:
    print(rag.responder(index, "Quais os principais problemas de telefonia?"))
except Exception as e:
    print("Geração requer GEMINI_API_KEY:", type(e).__name__)

## 7. Custos previstos (Gemini)

In [ ]:
tabela, *_ = custos.estimar("fewshot")
print(tabela)

## Conclusão
- Baseline TF-IDF já alcança **F1-macro 0.92**; o erro restante (~9%) é **ruído**.
- O LLM agrega na fatia ambígua e como **extrator estruturado confiável** (Pydantic).
- RAG recupera trechos relevantes (relevância@5 = 1.00) para perguntas de gestor.
- Ver `relatorio/` para análise de erro, insights e custos.